# Deploy Qwen2-Audio-7B-Instruct on Amazon SageMaker

This notebook demonstrates how to deploy the Qwen2-Audio-7B-Instruct model on Amazon SageMaker. Qwen2-Audio is a multimodal audio-language model capable of understanding both audio and text inputs.

## Features:
- Audio understanding and analysis
- Voice chat capabilities
- Scalable SageMaker endpoint deployment

## Prerequisites:
- AWS account with SageMaker permissions
- Sufficient quota for GPU instances (ml.g5.2xlarge or larger recommended)
- IAM role with necessary permissions

## 1. Setup and Dependencies

In [ ]:
!pip install -U sagemaker boto3 librosa soundfile huggingface_hub

In [ ]:
import boto3
import sagemaker
from sagemaker.huggingface import HuggingFaceModel, get_huggingface_llm_image_uri
from sagemaker.pytorch import PyTorchModel
from sagemaker import serializers, deserializers, image_uris
import json
import base64
import librosa
import numpy as np
from datetime import datetime
import time
import os

In [ ]:
# Initialize SageMaker session and role
sess = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sess.boto_region_name
bucket = sess.default_bucket()

print(f"SageMaker role: {role}")
print(f"SageMaker bucket: {bucket}")
print(f"SageMaker region: {region}")

## 2. Model Configuration

In [ ]:
# Model configuration
model_id = "Qwen/Qwen2-Audio-7B-Instruct"  # HuggingFace model ID
model_s3_uri = ""  # Set this to your S3 path if using S3 model (e.g., "s3://your-bucket/path/to/model/")
use_s3_model = True  # Set to True to use S3 model instead of HuggingFace Hub

# Example S3 model configuration:
# model_s3_uri = "s3://my-sagemaker-bucket/models/qwen2-audio-7b-instruct/"
# use_s3_model = True

endpoint_name = f"qwen2-audio-7b-instruct-{datetime.now().strftime('%Y-%m-%d-%H-%M-%S')}"

# Instance configuration - using GPU instance for audio processing
instance_type = "ml.g5.2xlarge"  # 1 A10G GPU, 24GB GPU memory
# For larger workloads, consider:
# instance_type = "ml.g5.4xlarge"  # 1 A10G GPU, 96GB system memory
# instance_type = "ml.p4d.2xlarge" # 1 A100 GPU, 40GB GPU memory

print(f"Model ID: {model_id}")
if use_s3_model:
    print(f"S3 Model URI: {model_s3_uri}")
    print("Using S3 model source")
else:
    print("Using HuggingFace Hub model source")
print(f"Endpoint name: {endpoint_name}")
print(f"Instance type: {instance_type}")

## 3. Create Model Inference Code

In [ ]:
# Create directory for model code
!mkdir -p code

In [ ]:
%%writefile code/inference.py
import json
import base64
import torch
import librosa
import numpy as np
import os
import boto3
import traceback
import sys
from io import BytesIO
import logging

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class ModelHandler:
    def __init__(self):
        self.model = None
        self.processor = None
        self.device = None
        self.s3_client = None
        self.model_loaded = False
    
    def download_from_s3(self, s3_uri, local_path):
        """Download model files from S3"""
        try:
            if not self.s3_client:
                self.s3_client = boto3.client('s3')
            
            # Parse S3 URI
            if not s3_uri.startswith('s3://'):
                raise ValueError("S3 URI must start with 's3://'")
            
            s3_parts = s3_uri[5:].split('/', 1)
            bucket = s3_parts[0]
            prefix = s3_parts[1] if len(s3_parts) > 1 else ''
            
            logger.info(f"Downloading model from S3: {s3_uri}")
            
            # Create local directory
            os.makedirs(local_path, exist_ok=True)
            
            # List objects in the S3 prefix
            paginator = self.s3_client.get_paginator('list_objects_v2')
            pages = paginator.paginate(Bucket=bucket, Prefix=prefix)
            
            file_count = 0
            for page in pages:
                if 'Contents' in page:
                    for obj in page['Contents']:
                        key = obj['Key']
                        # Skip directories
                        if key.endswith('/'):
                            continue
                        
                        # Get relative path
                        rel_path = key[len(prefix):].lstrip('/')
                        local_file_path = os.path.join(local_path, rel_path)
                        
                        # Create subdirectories if needed
                        local_dir = os.path.dirname(local_file_path)
                        if local_dir:
                            os.makedirs(local_dir, exist_ok=True)
                        
                        # Download file
                        logger.info(f"Downloading {key} to {local_file_path}")
                        self.s3_client.download_file(bucket, key, local_file_path)
                        file_count += 1
            
            logger.info(f"Model download completed. Downloaded {file_count} files.")
            return True
            
        except Exception as e:
            logger.error(f"Error downloading from S3: {str(e)}")
            logger.error(traceback.format_exc())
            raise
    
    def load_model(self):
        """Load the Qwen2-Audio model and processor"""
        try:
            logger.info("Starting Qwen2-Audio model loading...")
            
            # Set device
            self.device = "cuda" if torch.cuda.is_available() else "cpu"
            logger.info(f"Using device: {self.device}")
            
            # Log library versions
            import transformers
            logger.info(f"Transformers version: {transformers.__version__}")
            logger.info(f"Torch version: {torch.__version__}")
            
            # Log GPU info if available
            if torch.cuda.is_available():
                logger.info(f"CUDA version: {torch.version.cuda}")
                logger.info(f"GPU count: {torch.cuda.device_count()}")
                for i in range(torch.cuda.device_count()):
                    logger.info(f"GPU {i}: {torch.cuda.get_device_name(i)}")
                    logger.info(f"GPU {i} memory: {torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB")
            
            # Get model configuration from environment variables
            model_id = os.environ.get("HF_MODEL_ID", "Qwen/Qwen2-Audio-7B-Instruct")
            model_s3_uri = os.environ.get("MODEL_S3_URI", None)
            use_s3_model = os.environ.get("USE_S3_MODEL", "false").lower() == "true"
            
            logger.info(f"Model ID: {model_id}")
            logger.info(f"Use S3 Model: {use_s3_model}")
            if use_s3_model:
                logger.info(f"S3 Model URI: {model_s3_uri}")
            
            # Import transformers after logging
            logger.info("Importing transformers...")
            from transformers import Qwen2AudioForConditionalGeneration, AutoProcessor
            logger.info("Transformers imported successfully")
            
            if use_s3_model and model_s3_uri:
                # Download model from S3
                local_model_path = "/tmp/model"
                logger.info(f"Downloading model from S3 to {local_model_path}")
                self.download_from_s3(model_s3_uri, local_model_path)
                model_path = local_model_path
                logger.info(f"Using S3 model from: {model_s3_uri}")
            else:
                # Use HuggingFace Hub
                model_path = model_id
                logger.info(f"Using HuggingFace model: {model_id}")
            
            # Load processor first
            logger.info("Loading processor...")
            self.processor = AutoProcessor.from_pretrained(
                model_path,
                trust_remote_code=True,
                local_files_only=use_s3_model
            )
            logger.info("Processor loaded successfully")
            
            # Load model with more conservative settings
            logger.info("Loading model...")
            self.model = Qwen2AudioForConditionalGeneration.from_pretrained(
                model_path,
                device_map="auto",
                torch_dtype=torch.float16,
                trust_remote_code=True,
                local_files_only=use_s3_model,
                low_cpu_mem_usage=True,
                max_memory={0: "20GiB"}  # Limit GPU memory usage
            )
            logger.info("Model loaded successfully")
            
            # Set model to eval mode
            self.model.eval()
            
            self.model_loaded = True
            logger.info("✅ Qwen2-Audio model loading completed successfully")
            
        except Exception as e:
            logger.error(f"❌ Error loading model: {str(e)}")
            logger.error(f"Exception type: {type(e).__name__}")
            logger.error(traceback.format_exc())
            self.model_loaded = False
            raise
    
    def decode_audio(self, audio_data):
        """Decode base64 audio data"""
        try:
            logger.info("Decoding base64 audio data...")
            # Decode base64
            audio_bytes = base64.b64decode(audio_data)
            logger.info(f"Decoded audio bytes: {len(audio_bytes)}")
            
            # Load audio using librosa
            audio, sr = librosa.load(
                BytesIO(audio_bytes), 
                sr=self.processor.feature_extractor.sampling_rate
            )
            logger.info(f"Audio loaded: shape={audio.shape}, sr={sr}")
            
            return audio
            
        except Exception as e:
            logger.error(f"Error decoding audio: {str(e)}")
            logger.error(traceback.format_exc())
            raise
    
    def predict(self, data):
        """Generate prediction from input data"""
        try:
            if not self.model_loaded:
                raise RuntimeError("Model not loaded successfully")
                
            logger.info(f"Received prediction request: {type(data)}")
            logger.info(f"Request keys: {list(data.keys()) if isinstance(data, dict) else 'Not a dict'}")
            
            # Parse input
            conversation = data.get("conversation", [])
            max_length = data.get("max_length", 256)
            temperature = data.get("temperature", 0.7)
            top_p = data.get("top_p", 0.9)
            
            logger.info(f"Conversation length: {len(conversation)}")
            logger.info(f"Generation params: max_length={max_length}, temperature={temperature}, top_p={top_p}")
            
            if not conversation:
                raise ValueError("No conversation provided")
            
            # Process conversation and extract audio
            audios = []
            for i, message in enumerate(conversation):
                logger.info(f"Processing message {i}: role={message.get('role', 'unknown')}")
                if isinstance(message.get("content"), list):
                    for j, content_item in enumerate(message["content"]):
                        logger.info(f"Processing content item {j}: type={content_item.get('type', 'unknown')}")
                        if content_item.get("type") == "audio":
                            # Handle base64 encoded audio
                            if "audio_base64" in content_item:
                                logger.info("Processing base64 audio")
                                audio = self.decode_audio(content_item["audio_base64"])
                                audios.append(audio)
                            # Handle audio URL (for testing)
                            elif "audio_url" in content_item:
                                logger.info(f"Processing audio URL: {content_item['audio_url']}")
                                from urllib.request import urlopen
                                audio = librosa.load(
                                    BytesIO(urlopen(content_item["audio_url"]).read()),
                                    sr=self.processor.feature_extractor.sampling_rate
                                )[0]
                                audios.append(audio)
                                logger.info(f"Audio from URL loaded: shape={audio.shape}")
            
            logger.info(f"Total audio inputs: {len(audios)}")
            
            # Apply chat template
            logger.info("Applying chat template...")
            text = self.processor.apply_chat_template(
                conversation, 
                add_generation_prompt=True, 
                tokenize=False
            )
            logger.info(f"Chat template applied. Text length: {len(text)}")
            logger.info(f"Text preview: {text[:200]}...")
            
            # Process inputs
            logger.info("Processing inputs...")
            inputs = self.processor(
                text=text, 
                audios=audios if audios else None, 
                return_tensors="pt", 
                padding=True
            )
            logger.info(f"Input IDs shape: {inputs.input_ids.shape}")
            
            # Move to device
            logger.info(f"Moving inputs to device: {self.device}")
            inputs.input_ids = inputs.input_ids.to(self.device)
            if hasattr(inputs, 'audio_features') and inputs.audio_features is not None:
                inputs.audio_features = inputs.audio_features.to(self.device)
                logger.info(f"Audio features shape: {inputs.audio_features.shape}")
            
            # Generate response with compatible parameters
            logger.info("Starting generation...")
            with torch.no_grad():
                # Use compatible generation parameters for transformers 4.37.0
                generation_config = {
                    "max_length": max_length,
                    "temperature": temperature,
                    "top_p": top_p,
                    "do_sample": True,
                    "pad_token_id": self.processor.tokenizer.eos_token_id,
                    "eos_token_id": self.processor.tokenizer.eos_token_id,
                    "use_cache": True,
                    # Remove cache_position and other incompatible parameters
                }
                
                # Filter inputs to only pass compatible parameters
                model_inputs = {}
                if hasattr(inputs, 'input_ids') and inputs.input_ids is not None:
                    model_inputs['input_ids'] = inputs.input_ids
                if hasattr(inputs, 'attention_mask') and inputs.attention_mask is not None:
                    model_inputs['attention_mask'] = inputs.attention_mask.to(self.device)
                if hasattr(inputs, 'audio_features') and inputs.audio_features is not None:
                    model_inputs['audio_features'] = inputs.audio_features
                
                logger.info(f"Model input keys: {list(model_inputs.keys())}")
                
                generate_ids = self.model.generate(
                    **model_inputs,
                    **generation_config
                )
            
            logger.info(f"Generation completed. Output shape: {generate_ids.shape}")
            
            # Decode response
            generate_ids = generate_ids[:, inputs.input_ids.size(1):]
            response = self.processor.batch_decode(
                generate_ids, 
                skip_special_tokens=True, 
                clean_up_tokenization_spaces=False
            )[0]
            
            result = {
                "generated_text": response,
                "input_length": inputs.input_ids.size(1),
                "output_length": generate_ids.size(1)
            }
            
            logger.info(f"✅ Prediction completed successfully")
            logger.info(f"Generated text: {response}")
            return result
            
        except Exception as e:
            logger.error(f"❌ Prediction error: {str(e)}")
            logger.error(f"Exception type: {type(e).__name__}")
            logger.error(traceback.format_exc())
            raise

# Global model handler
model_handler = ModelHandler()

def model_fn(model_dir):
    """Load model for SageMaker"""
    try:
        logger.info(f"Model function called with model_dir: {model_dir}")
        logger.info(f"Environment variables:")
        for key, value in os.environ.items():
            if key.startswith(('HF_', 'SAGEMAKER_', 'MODEL_', 'USE_')):
                logger.info(f"  {key}={value}")
        
        logger.info("Loading model...")
        model_handler.load_model()
        logger.info("Model function completed successfully")
        return model_handler
        
    except Exception as e:
        logger.error(f"❌ Model function error: {str(e)}")
        logger.error(traceback.format_exc())
        raise

def input_fn(request_body, request_content_type):
    """Parse input data"""
    try:
        logger.info(f"Input function called with content_type: {request_content_type}")
        logger.info(f"Request body type: {type(request_body)}")
        logger.info(f"Request body length: {len(request_body) if hasattr(request_body, '__len__') else 'Unknown'}")
        
        if request_content_type == "application/json":
            parsed_data = json.loads(request_body)
            logger.info(f"✅ Input parsed successfully")
            return parsed_data
        else:
            raise ValueError(f"Unsupported content type: {request_content_type}")
            
    except Exception as e:
        logger.error(f"❌ Input function error: {str(e)}")
        logger.error(traceback.format_exc())
        raise

def predict_fn(input_data, model):
    """Generate prediction"""
    try:
        logger.info("Predict function called")
        result = model.predict(input_data)
        logger.info("✅ Predict function completed successfully")
        return result
        
    except Exception as e:
        logger.error(f"❌ Predict function error: {str(e)}")
        logger.error(traceback.format_exc())
        raise

def output_fn(prediction, accept):
    """Format output"""
    try:
        logger.info(f"Output function called with accept: {accept}")
        
        if accept == "application/json":
            result = json.dumps(prediction)
            logger.info("✅ Output function completed successfully")
            return result, accept
        else:
            raise ValueError(f"Unsupported accept type: {accept}")
            
    except Exception as e:
        logger.error(f"❌ Output function error: {str(e)}")
        logger.error(traceback.format_exc())
        raise

In [ ]:
%%writefile code/requirements.txt
git+https://github.com/huggingface/transformers
torch==2.6.0
librosa==0.11.0
soundfile==0.13.1
accelerate==1.9.0
numpy==1.26.4
boto3==1.40.1
huggingface_hub==0.34.0

%%writefile code/requirements.txt
transformers>=4.37.0
torch>=2.0.0
librosa>=0.10.0
soundfile>=0.12.0
accelerate>=0.20.0
numpy>=1.24.0
boto3>=1.26.0
huggingface_hub>=0.20.0

In [ ]:
# Helper function to upload model to S3
def upload_model_to_s3(local_model_path, s3_bucket, s3_prefix):
    """
    Upload a local model directory to S3
    
    Args:
        local_model_path: Local path to the model directory
        s3_bucket: S3 bucket name
        s3_prefix: S3 prefix/folder for the model
    
    Returns:
        S3 URI of the uploaded model
    """
    import os
    s3_client = boto3.client('s3')
    
    print(f"Uploading model from {local_model_path} to s3://{s3_bucket}/{s3_prefix}")
    
    # Walk through all files in the local model directory
    for root, dirs, files in os.walk(local_model_path):
        for file in files:
            local_file_path = os.path.join(root, file)
            # Get relative path from model directory
            rel_path = os.path.relpath(local_file_path, local_model_path)
            s3_key = os.path.join(s3_prefix, rel_path).replace('\\', '/')
            
            print(f"Uploading {local_file_path} to s3://{s3_bucket}/{s3_key}")
            s3_client.upload_file(local_file_path, s3_bucket, s3_key)
    
    s3_model_uri = f"s3://{s3_bucket}/{s3_prefix}"
    print(f"✅ Model uploaded successfully to: {s3_model_uri}")
    return s3_model_uri

# Example: Download and upload HuggingFace model to S3 using snapshot_download
def download_and_upload_hf_model_to_s3(model_id, s3_bucket, s3_prefix):
    """
    Download a HuggingFace model and upload it to S3 using snapshot_download
    
    Args:
        model_id: HuggingFace model ID
        s3_bucket: S3 bucket name  
        s3_prefix: S3 prefix for the model
    
    Returns:
        S3 URI of the uploaded model
    """
    from huggingface_hub import snapshot_download
    import tempfile
    
    print(f"Downloading {model_id} from HuggingFace Hub using snapshot_download...")
    
    with tempfile.TemporaryDirectory() as temp_dir:
        # Download entire model repository using snapshot_download
        local_model_path = snapshot_download(
            repo_id=model_id,
            cache_dir=temp_dir,
            local_dir=os.path.join(temp_dir, "model"),
            local_dir_use_symlinks=False,  # Copy files instead of symlinking
            ignore_patterns=["*.git*", "README.md", "*.md"],  # Skip unnecessary files
        )
        
        print(f"Model downloaded to: {local_model_path}")
        print(f"Model files:")
        for root, dirs, files in os.walk(local_model_path):
            for file in files[:10]:  # Show first 10 files
                file_path = os.path.join(root, file)
                rel_path = os.path.relpath(file_path, local_model_path)
                file_size = os.path.getsize(file_path) / (1024*1024)  # Size in MB
                print(f"  - {rel_path} ({file_size:.1f} MB)")
        
        # Upload to S3
        return upload_model_to_s3(local_model_path, s3_bucket, s3_prefix)

# Uncomment and run this section if you want to upload the model to S3, or you can upload your trained model to the s3 path directly.
UPLOAD_TO_S3 = True  # Set to True to upload
local_model_path = "" # Set your local model path
if UPLOAD_TO_S3:
    # s3_model_uri = download_and_upload_hf_model_to_s3(
    #     model_id=model_id,
    #     s3_bucket=bucket,
    #     s3_prefix=f"models/qwen2-audio-7b-instruct/{datetime.now().strftime('%Y-%m-%d-%H-%M-%S')}"
    # )

    s3_model_uri = upload_model_to_s3(
        local_model_path, 
        bucket, 
        s3_prefix=f"models/qwen2-audio-7b-instruct/{datetime.now().strftime('%Y-%m-%d-%H-%M-%S')}"
    )

    print(f"\n🎯 To use this S3 model:")
    print(f"   1. Set model_s3_uri = '{s3_model_uri}'")
    print(f"   2. Set use_s3_model = True")
    print(f"   3. Re-run the deployment cells")

print("To upload a model to S3, uncomment and configure the section above.")


## 5. Create and Deploy SageMaker Model

### Important: Container Selection for Multimodal Models

**Issue**: The HuggingFace LLM DLC doesn't support `qwen2_audio` model type because:
- It's designed for text-only language models using `text-generation-server`
- Uses `text-generation-server` which doesn't support multimodal models
- Lacks audio processing capabilities
- Qwen2-Audio tokenizer missing `bos_token_id` causes `RuntimeError: Could not infer dtype of NoneType`

**Solution**: We use PyTorch DLC instead because:
- ✅ Supports custom model architectures (including multimodal)
- ✅ Allows custom inference code with audio processing
- ✅ Full control over model loading and inference pipeline
- ✅ Compatible with transformers library for Qwen2-Audio

In [ ]:
# Get the appropriate container image for multimodal models
# Since Qwen2-Audio is not supported by the HuggingFace LLM DLC, we'll use the standard PyTorch DLC

image_uri = f"763104351884.dkr.ecr.{region}.amazonaws.com/pytorch-inference:2.6.0-gpu-py312-cu124-ubuntu22.04-sagemaker"

print(f"Using PyTorch DLC image: {image_uri}")
print("Note: Using PyTorch DLC instead of HuggingFace LLM DLC due to multimodal model requirements")

In [ ]:
# Create SageMaker Model with PyTorch container for multimodal support
from sagemaker.pytorch import PyTorchModel
from sagemaker import Model
import tarfile

env_vars = {
    "HF_MODEL_ID": model_id,
    "SAGEMAKER_CONTAINER_LOG_LEVEL": "20",
    "SAGEMAKER_REGION": region,
    "TRANSFORMERS_CACHE": "/tmp/transformers_cache",
    "HF_HOME": "/tmp/huggingface",
    "TORCH_HOME": "/tmp/torch"
}

# Add S3 configuration if using S3 model
if use_s3_model and model_s3_uri:
    env_vars.update({
        "MODEL_S3_URI": model_s3_uri,
        "USE_S3_MODEL": "true"
    })
    print(f"Configured for S3 model loading from: {model_s3_uri}")

# Create a dummy model.tar.gz to satisfy PyTorchModel requirements
def create_dummy_model_artifact():
    import os
    import tempfile
    
    with tempfile.TemporaryDirectory() as temp_dir:
        # Create a dummy model file
        dummy_model_path = os.path.join(temp_dir, "model.pth")
        with open(dummy_model_path, "w") as f:
            f.write("# Dummy model file - actual model loaded in inference.py")
        
        # Create tar.gz
        model_tar_path = os.path.join(temp_dir, "model.tar.gz")
        with tarfile.open(model_tar_path, "w:gz") as tar:
            tar.add(dummy_model_path, arcname="model.pth")
        
        # Upload to S3
        s3_model_path = f"s3://{bucket}/dummy-model/model.tar.gz"
        boto3.client('s3').upload_file(model_tar_path, bucket, "dummy-model/model.tar.gz")
        print(f"Created dummy model artifact at: {s3_model_path}")
        return s3_model_path

# Create dummy model artifact
dummy_model_s3_path = create_dummy_model_artifact()

# Use PyTorchModel with dummy model artifact
pytorch_model = PyTorchModel(
    name=endpoint_name.replace('-', ''),
    model_data=dummy_model_s3_path,  # Use dummy model artifact
    source_dir="code",
    entry_point="inference.py",
    image_uri=image_uri,
    role=role,
    env=env_vars,
    sagemaker_session=sess,
    py_version="py312",
    framework_version="2.6.0"
)

print(f"Created PyTorch model: {pytorch_model.name}")
print(f"Dummy model artifact: {dummy_model_s3_path}")
print(f"Environment variables: {env_vars}")
print("Using PyTorch container with custom model loading for Qwen2-Audio multimodal model")

In [ ]:
# Deploy the model to a SageMaker endpoint
print(f"Deploying model to endpoint: {endpoint_name}")
print(f"Instance type: {instance_type}")
print("This may take 10-15 minutes...")

predictor = pytorch_model.deploy(
    initial_instance_count=1,
    instance_type=instance_type,
    endpoint_name=endpoint_name,
    container_startup_health_check_timeout=900,  # 15 minutes for multimodal model loading
    serializer=serializers.JSONSerializer(),
    deserializer=deserializers.JSONDeserializer()
)

print(f"✅ Model deployed successfully to endpoint: {endpoint_name}")

## 6. Test the Deployed Model

### Test 1: Voice Chat Mode (Audio-only input)

In [ ]:
# Test voice chat mode with audio URL (for demonstration)
voice_chat_payload = {
    "conversation": [
        {
            "role": "user", 
            "content": [
                {
                    "type": "audio", 
                    "audio_url": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen2-Audio/audio/guess_age_gender.wav"
                },
                {
                    "type": "text",
                    "text": "Guess the age and gender of the speaker."
                }
            ]
        }
    ],
    "max_length": 256,
    "temperature": 0.7,
    "top_p": 0.9
}

print("Testing voice chat mode...")
try:
    response = predictor.predict(voice_chat_payload)
    print(f"Response: {response['generated_text']}")
    print(f"Input length: {response['input_length']}")
    print(f"Output length: {response['output_length']}")
except Exception as e:
    print(f"Error: {e}")

### Test 2: Audio Analysis Mode (Audio + Text input)

In [ ]:
# Test audio analysis mode
audio_analysis_payload = {
    "conversation": [
        {
            "role": "system", 
            "content": "You are a helpful assistant."
        },
        {
            "role": "user", 
            "content": [
                {
                    "type": "audio", 
                    "audio_url": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen2-Audio/audio/glass-breaking-151256.mp3"
                },
                {
                    "type": "text", 
                    "text": "What's that sound?"
                }
            ]
        }
    ],
    "max_length": 256,
    "temperature": 0.7
}

print("Testing audio analysis mode...")
try:
    response = predictor.predict(audio_analysis_payload)
    print(f"Response: {response['generated_text']}")
    print(f"Input length: {response['input_length']}")
    print(f"Output length: {response['output_length']}")
except Exception as e:
    print(f"Error: {e}")

### Test 3: Local Audio File with Base64 Encoding

In [ ]:
def encode_audio_to_base64(audio_path):
    """Encode local audio file to base64"""
    with open(audio_path, "rb") as audio_file:
        audio_bytes = audio_file.read()
        audio_base64 = base64.b64encode(audio_bytes).decode('utf-8')
    return audio_base64

# If you have a local audio file, uncomment and modify this section:
local_audio_path = '' # a local wav file
if os.path.exists(local_audio_path):
    audio_base64 = encode_audio_to_base64(local_audio_path)
    
    local_audio_payload = {
        "conversation": [
            {
                "role": "user", 
                "content": [
                    {
                        "type": "audio", 
                        "audio_base64": audio_base64
                    },
                    {
                        "type": "text", 
                        "text": "Please describe what you hear in this audio."
                    }
                ]
            }
        ],
        "max_length": 256
    }
    
    print("Testing with local audio file...")
    response = predictor.predict(local_audio_payload)
    print(f"Response: {response['generated_text']}")

print("To test with local audio files, uncomment and modify the code above.")

## 6. Performance Monitoring

In [ ]:
# Monitor endpoint performance
import boto3
from datetime import datetime, timedelta

cloudwatch = boto3.client('cloudwatch')

def get_endpoint_metrics(endpoint_name, start_time, end_time):
    """Get CloudWatch metrics for the endpoint"""
    
    metrics = {
        'Invocations': 'Sum',
        'ModelLatency': 'Average', 
        'OverheadLatency': 'Average',
        'Invocation4XXErrors': 'Sum',
        'Invocation5XXErrors': 'Sum'
    }
    
    results = {}
    
    for metric_name, statistic in metrics.items():
        try:
            response = cloudwatch.get_metric_statistics(
                Namespace='AWS/SageMaker',
                MetricName=metric_name,
                Dimensions=[
                    {
                        'Name': 'EndpointName',
                        'Value': endpoint_name
                    },
                ],
                StartTime=start_time,
                EndTime=end_time,
                Period=300,  # 5 minutes
                Statistics=[statistic]
            )
            
            if response['Datapoints']:
                results[metric_name] = response['Datapoints'][-1][statistic]
            else:
                results[metric_name] = 0
                
        except Exception as e:
            print(f"Error getting {metric_name}: {e}")
            results[metric_name] = "N/A"
    
    return results

# Get metrics for the last hour
end_time = datetime.utcnow()
start_time = end_time - timedelta(hours=1)

print("Endpoint Performance Metrics (Last Hour):")
metrics = get_endpoint_metrics(endpoint_name, start_time, end_time)

for metric, value in metrics.items():
    if isinstance(value, (int, float)):
        if 'Latency' in metric:
            print(f"  {metric}: {value:.2f} ms")
        else:
            print(f"  {metric}: {value}")
    else:
        print(f"  {metric}: {value}")

## 7. Batch Processing Example

In [ ]:
# Example of batch processing multiple audio inputs
batch_payload = {
    "conversation": [
        {
            "role": "user", 
            "content": [
                {
                    "type": "audio", 
                    "audio_url": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen2-Audio/audio/glass-breaking-151256.mp3"
                },
                {
                    "type": "text", 
                    "text": "What's that sound?"
                }
            ]
        },
        {
            "role": "assistant", 
            "content": "It is the sound of glass shattering."
        },
        {
            "role": "user", 
            "content": [
                {
                    "type": "audio", 
                    "audio_url": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen2-Audio/audio/f2641_0_throatclearing.wav"
                },
                {
                    "type": "text", 
                    "text": "What can you hear?"
                }
            ]
        }
    ],
    "max_length": 256
}

print("Testing batch processing with conversation history...")
try:
    response = predictor.predict(batch_payload)
    print(f"Response: {response['generated_text']}")
except Exception as e:
    print(f"Error: {e}")

## 8. Cleanup Resources

In [ ]:
# # ⚠️ WARNING: This will delete your endpoint and stop billing
# # Only run this when you're done testing

# cleanup = False  # Set to True to cleanup resources

# if cleanup:
#     print(f"🗑️  Deleting endpoint: {endpoint_name}")
#     try:
#         predictor.delete_endpoint(delete_endpoint_config=True)
#         print("✅ Endpoint deleted successfully")
#     except Exception as e:
#         print(f"❌ Error deleting endpoint: {e}")
# else:
#     print("⚠️  Endpoint is still running. To delete it, set cleanup=True and run this cell.")
#     print(f"   Current endpoint: {endpoint_name}")
#     print(f"   Estimated cost: ~$1.20-2.03/hour depending on instance type")